## Attention-Head Activation Analysis

In [1]:
%load_ext autoreload
%autoreload 2

### Overview

Pure activation-reading notebook — no patching or generation. For each attention head, projects its individual contribution to the residual stream at the prediction position and reads it two ways: a logit lens (the probability it assigns to the operand/output tokens) and a residual-similarity score (how closely it aligns with the model's own in-context representations of those tokens).

### Set-up

Imports `_config`/`_dataset`/`_prompt`/`_mapping` plus the activation-caching helpers used below.

In [2]:
import torch
import gc
import random
import pandas as pd
from tqdm import tqdm

import sys
sys.path.append("src")
import _config
import _dataset
import _prompt
import _mapping
from _intervention import prepare_batch_multitoken_intervention, batch_intervene, get_attention_freeze_hooks, prepare_batch_multitoken_head_intervention

## Experiment Config

In [3]:
prompt_config = _config.PromptConfig(
    model_type="GPT-OSS_stepwise", # GPT-OSS or R1
    prompt_type="h_pre_penultimate_sum", # {null / h / h1 / h2}_{null / pre_result / pre_final_sum / ...}
)
intervention_config = _config.InterventionConfig(
    intervention_loc="", # restatement or reasoning or restatement_and_reasoning
    intervention_ids=[20],
    tok_pos_fn=_mapping.intervene_id_to_tok_pos_stepwise_3_digit_h,
)
attention_config = _config.AttentionFreezeConfig(
    enabled=False,
    num_attention=20,
    dataset_fn=_dataset.create_h_dataset,
    num_digits=3,
    prompt_fn=_prompt.get_stepwise_prompt,
    divide_num=22,
)
run_config = _config.RunConfig(
    experiment_root="experiments/attention_heads",
    result_dir="head_activation",
)

tok_pos_fn = intervention_config.tok_pos_fn
prompt_fn = attention_config.prompt_fn
num_attention = attention_config.num_attention
freeze_attention = attention_config.enabled
modifier_fn = lambda prompt, add_ds_entry: prompt

intervene = True
if intervene:
    intervene_str = "intervened_"
else:
    intervene_str = ""

output_tok_pos = -1

## Set up Experiment

Loads the model/tokenizer, builds the (here disabled) attention-freeze hooks, loads the configured base/source prompts, and resolves the token position to read activations at.

In [4]:
model, tokenizer = _config.load_model(prompt_config.model_type)

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [5]:
def intervene_on_final_sum(prompt, add_ds_entry):
    source_prompt = attention_config.prompt_fn(
        add_ds_entry["source_1_digits"],
        add_ds_entry["source_2_digits"],
        add_ds_entry["source_1_num"],
        add_ds_entry["source_2_num"],
    )
    return _prompt.get_intervened_prompt(intervention_ids, prompt, source_prompt)

modifier_fn = intervene_on_final_sum

In [6]:
attention_freeze_hooks = _config.build_attention_freeze_hooks(
    model,
    tokenizer,
    attention_config,
    modifier_fn=modifier_fn,
)

In [7]:
prompts = _config.load_prompts(prompt_config)
print(f"loaded {len(prompts)} prompts")

loaded 256 prompts


In [8]:
intervention_ids = _config.resolve_intervention_ids(
    prompt_config.model_type,
    prompt_config.prompt_type,
    intervention_config.intervention_loc,
    intervention_config.intervention_ids,
)
print(intervention_ids)

[20]


In [9]:
# for i, row in prompts.iterrows():
#     print(list(enumerate(tokenizer.convert_ids_to_tokens(tokenizer(row["base_prompt"], add_special_tokens=False, return_tensors="pt")["input_ids"][0]))))

In [9]:
# Only the last-token position is patched: that is the position whose attention-
# head output feeds directly into the next-token prediction.
tok_pos_list = [intervention_config.tok_pos_fn[attention_config.divide_num] - 1]
print(tok_pos_list)

[240]


## Logit Lens

For each batch, projects every head's individual contribution to the residual stream through the unembedding and reads the probability assigned to the operand and output tokens.

In [10]:
from _intervention import forward_with_cache, get_logit_lens

batch_size = 24

num_layers = len(model.model.layers)
num_heads  = model.config.num_attention_heads
head_dim   = model.config.hidden_size // num_heads

print(num_layers, num_heads, head_dim)

num_layers, num_heads, head_dim = 24, 64, 64

# Cache the input to o_proj (= concatenated head outputs) for all layers in one pass
module_names = [f"model.layers.{layer}.self_attn.o_proj" for layer in range(num_layers)]

# Set up output CSV
header   = list(prompts.columns) + ['layer', 'head_num', 'operand_1_1_prob', 'operand_1_2_prob', 'operand_2_prob', 'factual_output_prob']
logit_lens_run_config = _config.RunConfig(
    experiment_root=run_config.experiment_root,
    result_dir=run_config.result_dir,
    output_filename=f"{intervene_str}logit_lens{prompt_config.suffix}_{output_tok_pos}.csv",
)
filepath = _config.build_run_output_filepath(prompt_config, logit_lens_run_config, header)

for i in tqdm(range(0, len(prompts), batch_size)):
    batch_rows     = prompts.iloc[i:i+batch_size]
    cur_batch_size = len(batch_rows)
    if intervene:
        input_prompts = [_prompt.get_intervened_prompt(intervention_ids, row['base_prompt'], row['source_prompt']) for _, row in batch_rows.iterrows()]
    else:
        input_prompts = batch_rows['base_prompt'].tolist()
    

    # Extract numbers at intervention ids 19, 20, and 21 from each base prompt via divide_prompt,
    # then take the first token (handles multi-token numbers)
    id19_ids     = tokenizer([_prompt.divide_prompt(19, p)[1] for p in input_prompts],
                             add_special_tokens=False, return_tensors="pt")["input_ids"][:, 0]
    id20_ids     = tokenizer([_prompt.divide_prompt(20, p)[1] for p in input_prompts],
                             add_special_tokens=False, return_tensors="pt")["input_ids"][:, 0]
    id21_ids     = tokenizer([_prompt.divide_prompt(21, p)[1] for p in input_prompts],
                             add_special_tokens=False, return_tensors="pt")["input_ids"][:, 0]
    factual_ids  = tokenizer([str(n) for n in batch_rows['factual_output'].tolist()],
                             add_special_tokens=False, return_tensors="pt")["input_ids"][:, 0]

    tokens = tokenizer(input_prompts, add_special_tokens=False, return_tensors="pt",
                            padding=True, padding_side="left").to(model.device)

    # Single forward pass — cache inputs to o_proj at every layer
    _, cache = forward_with_cache(model, tokens["input_ids"],
                                  module_names=module_names,
                                  attention_mask=tokens["attention_mask"],
                                  pre_hook=True)

    for layer in range(num_layers):
        # o_proj input: [batch, seq_len, num_heads * head_dim]
        o_proj_input = cache[f"model.layers.{layer}.self_attn.o_proj"]

        # Per-head vectors at position -1: [batch, num_heads, head_dim]
        per_head = o_proj_input[:, output_tok_pos, :].view(cur_batch_size, num_heads, head_dim)

        # Project all heads through W_o in one batched einsum
        # W_o: [hidden_size, num_heads * head_dim]
        # Reshape → [num_heads, head_dim, hidden_size] so each head's slice is contiguous
        W_o       = model.model.layers[layer].self_attn.o_proj.weight   # [hidden_size, num_heads * head_dim]
        W_o_heads = W_o.reshape(model.config.hidden_size, num_heads, head_dim).permute(1, 2, 0)  # [num_heads, head_dim, hidden_size]

        # head_contributions: [batch, num_heads, hidden_size]
        head_contributions = torch.einsum('bnh,nho->bno', per_head, W_o_heads)

        # Logit lens: flatten → [batch * num_heads, hidden_size], apply norm + lm_head
        flat   = head_contributions.reshape(cur_batch_size * num_heads, model.config.hidden_size)
        logits = get_logit_lens(model, flat)                                    # [batch * num_heads, vocab_size]
        probs  = torch.softmax(logits, dim=-1).reshape(cur_batch_size, num_heads, -1)  # [batch, num_heads, vocab_size]

        # Gather probabilities for the three label tokens: results are [batch, num_heads]
        dev = probs.device
        i19 = id19_ids.to(dev).unsqueeze(1).expand(-1, num_heads)
        i20 = id20_ids.to(dev).unsqueeze(1).expand(-1, num_heads)
        i21 = id21_ids.to(dev).unsqueeze(1).expand(-1, num_heads)
        fo  = factual_ids.to(dev).unsqueeze(1).expand(-1, num_heads)

        operand_1_1_probs      = probs.gather(2, i19.unsqueeze(2)).squeeze(2)  # [batch, num_heads]
        operand_1_2_probs      = probs.gather(2, i20.unsqueeze(2)).squeeze(2)
        operand_2_probs      = probs.gather(2, i21.unsqueeze(2)).squeeze(2)
        factual_output_probs = probs.gather(2, fo.unsqueeze(2)).squeeze(2)

        # Write one CSV row per (prompt, head)
        for j, (_, row) in enumerate(batch_rows.iterrows()):
            for head in range(num_heads):
                _config.write_to_csv(filepath, row.to_list() + [
                    layer, head,
                    operand_1_1_probs[j, head].item(),
                    operand_1_2_probs[j, head].item(),
                    operand_2_probs[j, head].item(),
                    factual_output_probs[j, head].item()
                ])

        del o_proj_input, per_head, W_o_heads, head_contributions, flat, logits, probs
        torch.cuda.empty_cache()

    del cache, tokens
    torch.cuda.empty_cache()
    gc.collect()

24 64 45


100%|████████████████████████████████████████████████████████████████████| 11/11 [18:17<00:00, 99.76s/it]


## Residual Similarity

For each batch, compares each head's projected contribution at the prediction position against the model's own in-context representations of the operand and output tokens, via cosine similarity and a signed projection score.

In [10]:
# EDIT THIS CELL
# Score each attention-head output by its cosine similarity to the *in-context* residual-
# stream representation of operand 1, operand 2, and the factual-output token, taken from
# a single forward pass.
#
#  - Operand 1 / operand 2 references: the residual stream AT each operand's own token
#    position (positions are obtained from `tok_pos_fn[20]` / `tok_pos_fn[21]`). At that
#    position, the model has just read the operand, so the rep is "this number, in this
#    Harmony prompt, in this layer".
#  - Factual-output reference: the residual stream AT the factual-output token. Since
#    factual_output is what the model would generate next, we APPEND its first token to
#    each base prompt, do a single forward pass on the extended sequence, and read the
#    appended position.
#  - Head being scored: read at the prediction site, which after the append is at -2
#    (the original last token of base_prompt).

from _intervention import forward_with_cache
import torch.nn.functional as F

batch_size = 24

num_layers = len(model.model.layers)
num_heads  = model.config.num_attention_heads
head_dim   = model.config.hidden_size // num_heads

print(num_layers, num_heads, head_dim)

num_layers, num_heads, head_dim = 24, 64, 64

# Token positions in the *extended* (base_prompt + factual_first_tok) sequence.
# `output_tok_pos` is the prediction site within the original base prompt. Because we
# append exactly one token, a negative output_tok_pos shifts left by 1; a positive
# absolute index is unchanged.
prediction_pos  = output_tok_pos - 1 if output_tok_pos < 0 else output_tok_pos
factual_ref_pos = -1
op11_pos         = tok_pos_fn[19]
op12_pos         = tok_pos_fn[20]
op2_pos         = tok_pos_fn[21]

# One forward pass caches both o_proj inputs (per-head outputs) and layer inputs
# (residual stream) at every layer.
o_proj_modules = [f"model.layers.{layer}.self_attn.o_proj" for layer in range(num_layers)]
layer_modules  = [f"model.layers.{layer}"                  for layer in range(num_layers)]
module_names   = o_proj_modules + layer_modules

header   = list(prompts.columns) + [
    'layer', 'head_num',
    'operand_1_1_cos_sim',  'operand_1_2_cos_sim',  'operand_2_cos_sim',  'factual_output_cos_sim',
    'operand_1_1_proj',     'operand_1_2_proj',     'operand_2_proj',     'factual_output_proj',
]
residual_similarity_run_config = _config.RunConfig(
    experiment_root=run_config.experiment_root,
    result_dir=run_config.result_dir,
    output_filename=f"{intervene_str}residual_similarity{prompt_config.suffix}_{output_tok_pos}.csv",
)
filepath = _config.build_run_output_filepath(prompt_config, residual_similarity_run_config, header)

for i in tqdm(range(0, len(prompts), batch_size)):
    batch_rows     = prompts.iloc[i:i+batch_size]
    cur_batch_size = len(batch_rows)
    # Match the logit-lens cell: when `intervene` is True, swap in source numbers at
    # `intervention_ids` (prompt-level intervention) before tokenizing.
    if intervene:
        input_prompts = [_prompt.get_intervened_prompt(intervention_ids, row['base_prompt'], row['source_prompt']) for _, row in batch_rows.iterrows()]
    else:
        input_prompts = batch_rows['base_prompt'].tolist()

    base_tokens = tokenizer(input_prompts, add_special_tokens=False, return_tensors="pt",
                            padding=True, padding_side="left").to(model.device)

    # Append the first token of str(factual_output) to each row so the factual-output
    # rep can be read in-context at position -1. Using only the first token keeps every
    # row the same length (factual_output can be multi-token for 4-digit sums).
    factual_first_tok = tokenizer(
        [str(n) for n in batch_rows['factual_output'].tolist()],
        add_special_tokens=False, return_tensors="pt",
    )["input_ids"][:, 0].to(model.device)

    extended_input_ids = torch.cat(
        [base_tokens["input_ids"], factual_first_tok.unsqueeze(1)], dim=1)
    extended_attention_mask = torch.cat([
        base_tokens["attention_mask"],
        torch.ones(cur_batch_size, 1,
                   dtype=base_tokens["attention_mask"].dtype, device=model.device),
    ], dim=1)

    _, cache = forward_with_cache(model, extended_input_ids,
                                  module_names=module_names,
                                  attention_mask=extended_attention_mask,
                                  pre_hook=True)

    for layer in range(num_layers):
        # Per-head outputs at the prediction site (one before the appended factual token).
        o_proj_input = cache[f"model.layers.{layer}.self_attn.o_proj"]                     # [batch, seq, n_h * d_h]
        per_head     = o_proj_input[:, prediction_pos, :].view(cur_batch_size, num_heads, head_dim)

        # Project each head through its slice of W_o so the contribution lives in
        # residual-stream space.
        W_o       = model.model.layers[layer].self_attn.o_proj.weight                       # [hidden, n_h * d_h]
        W_o_heads = W_o.reshape(model.config.hidden_size, num_heads, head_dim).permute(1, 2, 0)  # [n_h, d_h, hidden]
        head_contributions = torch.einsum('bnh,nho->bno', per_head, W_o_heads)              # [batch, n_h, hidden]

        # In-context residual-stream reps at this layer for the three reference positions.
        residual    = cache[f"model.layers.{layer}"]                                        # [batch, seq, hidden]
        rep_op11     = residual[:, op11_pos,         :].to(head_contributions.device).to(head_contributions.dtype)
        rep_op12     = residual[:, op12_pos,         :].to(head_contributions.device).to(head_contributions.dtype)
        rep_op2     = residual[:, op2_pos,         :].to(head_contributions.device).to(head_contributions.dtype)
        rep_factual = residual[:, factual_ref_pos, :].to(head_contributions.device).to(head_contributions.dtype)

        # Cosine similarity (direction only): [batch, n_h, hidden] vs [batch, 1, hidden] -> [batch, n_h].
        op11_cos = F.cosine_similarity(head_contributions, rep_op11.unsqueeze(1),     dim=-1)
        op12_cos = F.cosine_similarity(head_contributions, rep_op12.unsqueeze(1),     dim=-1)
        op2_cos = F.cosine_similarity(head_contributions, rep_op2.unsqueeze(1),     dim=-1)
        fo_cos  = F.cosine_similarity(head_contributions, rep_factual.unsqueeze(1), dim=-1)

        # Signed projection coefficient beta = (h . r) / ||r||^2: scalar in the
        # decomposition h = beta * r + h_perp. Magnitude-aware analogue of cosine
        # similarity; ~ A[pred_pos, ref_pos] for clean copy heads, but factors in
        # the OV circuit and can be negative.
        eps      = 1e-12
        op11_sq   = (rep_op11     * rep_op11    ).sum(dim=-1).clamp_min(eps).unsqueeze(1)   # [batch, 1]
        op12_sq   = (rep_op12     * rep_op12    ).sum(dim=-1).clamp_min(eps).unsqueeze(1)
        op2_sq   = (rep_op2     * rep_op2    ).sum(dim=-1).clamp_min(eps).unsqueeze(1)
        fact_sq  = (rep_factual * rep_factual).sum(dim=-1).clamp_min(eps).unsqueeze(1)
        op11_proj = (head_contributions * rep_op11    .unsqueeze(1)).sum(dim=-1) / op11_sq   # [batch, n_h]
        op12_proj = (head_contributions * rep_op12    .unsqueeze(1)).sum(dim=-1) / op12_sq
        op2_proj = (head_contributions * rep_op2    .unsqueeze(1)).sum(dim=-1) / op2_sq
        fo_proj  = (head_contributions * rep_factual.unsqueeze(1)).sum(dim=-1) / fact_sq

        for j, (_, row) in enumerate(batch_rows.iterrows()):
            for head in range(num_heads):
                _config.write_to_csv(filepath, row.to_list() + [
                    layer, head,
                    op11_cos[j, head].item(),
                    op12_cos[j, head].item(),
                    op2_cos[j, head].item(),
                    fo_cos[j, head].item(),
                    op11_proj[j, head].item(),
                    op12_proj[j, head].item(),
                    op2_proj[j, head].item(),
                    fo_proj[j, head].item(),
                ])

        del o_proj_input, per_head, W_o_heads, head_contributions, residual, rep_op11, rep_op12, rep_op2, rep_factual
        del op11_cos, op12_cos, op2_cos, fo_cos, op11_sq, op12_sq, op2_sq, fact_sq, op11_proj, op12_proj, op2_proj, fo_proj
        torch.cuda.empty_cache()

    del cache, base_tokens, extended_input_ids, extended_attention_mask, factual_first_tok
    torch.cuda.empty_cache()
    gc.collect()

24 64 45


100%|█████████████████████████████████████████████████████████████████████████████| 11/11 [16:11<00:00, 88.35s/it]
